# Belege (Extraction) Import via API

Imports data from `belege_*.xlsx` files into the `Extraction` model
using HTTP POST calls to the motm REST API.

**API base URL**: `http://localhost:8000/motm/api/`

Columns in the xlsx (row index 2 = header):
- `beleg_id` → `Extraction.identifier`
- `quelle` → source reference (informational)
- `interview_id` → linked `Interview.archive_id`
- `interview_datum` → (informational)
- `quelle_sprecher` → speaker `Person.identifier` → sets `Interview.interviewee`
- `betrifft_personen` → `Extraction.people_mentioned` (Person identifier)
- `timecode` → `Extraction.timecode`
- `themen` → `Extraction.concepts` (Concept labels, comma-separated → M2M)
- `zitat` → `Extraction.quote`
- `markierung` → `Extraction.classification`
- `event_ids` → `Extraction.event` (comma-separated event IDs)
- `event_ids_confidence` → `Extraction.event_confidence`
- `notizen` → `Extraction.notes`

In [1]:
import os
from pathlib import Path

import requests
from openpyxl import load_workbook
from tqdm.auto import tqdm
from fast_langdetect import detect as detect_language

/home/mapto/work/memorymap-toolkit-motm/notebook/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import getpass

BASE_URL = os.environ.get("MMT_API_URL", "http://localhost:8000/motm/api")
LOGIN_URL = os.environ.get("MMT_LOGIN_URL", "http://localhost:8000/admin/login/")
SESSION = requests.Session()

DATA_DIR = Path.cwd().parent / "data" / "schede mappatura"

# Authenticate via Django admin session
# username = input("Admin username: ")
# password = getpass.getpass("Admin password: ")
SESSION.get(LOGIN_URL)
SESSION.post(LOGIN_URL, data={
    "username": "admin",
    "password": "admin",
    "csrfmiddlewaretoken": SESSION.cookies["csrftoken"],
    "next": "/admin/",
})
SESSION.headers["X-CSRFToken"] = SESSION.cookies.get("csrftoken", "")
SESSION.headers["Referer"] = LOGIN_URL
print("Authenticated" if SESSION.cookies.get("sessionid") else "Login failed")

Authenticated


## API helpers

In [3]:
def is_empty(value):
    return value in [None, "", "-", "–", "—"]


def clean(value):
    if value is None:
        return ""
    return str(value).strip()


def api_get(endpoint, params=None):
    """GET from the API, return full JSON list (handles pagination)."""
    if params is None:
        params = {}
    params.setdefault("limit", 10000)
    resp = SESSION.get(f"{BASE_URL}/{endpoint}/", params=params)
    resp.raise_for_status()
    data = resp.json()
    if isinstance(data, dict) and "results" in data:
        return data["results"]
    return data


def api_post(endpoint, payload):
    """POST to the API, return created object as dict."""
    resp = SESSION.post(f"{BASE_URL}/{endpoint}/", json=payload)
    resp.raise_for_status()
    return resp.json()


def api_patch(endpoint, obj_id, payload):
    """PATCH an existing object via the API, return updated object as dict."""
    resp = SESSION.patch(f"{BASE_URL}/{endpoint}/{obj_id}/", json=payload)
    resp.raise_for_status()
    return resp.json()

## Lookup helpers

In [4]:
import re

def get_person_id(identifier):
    """Find a Person by identifier. Returns id or None."""
    if is_empty(identifier):
        return None
    existing = api_get("persons", params={"search": identifier})
    match = next((p for p in existing if p.get("identifier") == identifier), None)
    if match:
        return match["id"]
    return None


def parse_interview_id_from_quelle(quelle):
    """Extract the interview ID from a quelle Markdown link.

    The quelle column format is:
        Interviewer Name – [IS_E_00124](https://dgd.ids-mannheim.de/...)
    Returns the link caption (e.g. 'IS_E_00124') or None.
    """
    if is_empty(quelle):
        return None
    m = re.search(r"\[([A-Z]+_[A-Z]_\d+)\]", quelle)
    if m:
        return m.group(1)
    return None


def get_interview_id(archive_id, quelle=None):
    """Find an Interview by archive_id. Falls back to parsing quelle if
    archive_id is empty or not found in the database. Returns id or None."""
    if is_empty(archive_id) and quelle:
        archive_id = parse_interview_id_from_quelle(quelle)
    if is_empty(archive_id):
        return None
    existing = api_get("interviews", params={"search": archive_id})
    match = next((i for i in existing if i["archive_id"] == archive_id), None)
    if match:
        return match["id"]
    # archive_id from the explicit column didn't match; try quelle as fallback
    if quelle:
        fallback_id = parse_interview_id_from_quelle(quelle)
        if fallback_id and fallback_id != archive_id:
            existing = api_get("interviews", params={"search": fallback_id})
            match = next((i for i in existing if i["archive_id"] == fallback_id), None)
            if match:
                return match["id"]
    return None


# Local cache: lowercase label → concept id
_concept_cache = {}

def get_or_create_concept(label):
    """Get or create a Concept by label (case-insensitive). Returns id."""
    label = label.strip()
    if is_empty(label):
        return None
    key = label.lower()
    if key in _concept_cache:
        return _concept_cache[key]
    existing = api_get("concepts", params={"search": label})
    match = next((c for c in existing if c["label"].lower() == key), None)
    if match:
        _concept_cache[key] = match["id"]
        return match["id"]
    created = api_post("concepts", {"label": label})
    _concept_cache[key] = created["id"]
    return created["id"]

## Read and import a belege xlsx

In [5]:
def import_belege(xlsx_path, sheet_name="Belege"):
    """Import all rows from a belege xlsx file via the API."""
    wb = load_workbook(xlsx_path)
    sheet = wb[sheet_name]

    # Count data rows (skip first 3: title, blank, header)
    all_rows = list(sheet.iter_rows(values_only=True))
    data_rows = [r for r in all_rows[3:] if r and not all(cell is None for cell in r)]

    results = []
    errors = []

    for row in tqdm(data_rows, desc=xlsx_path.stem, unit="row"):
        cells = (list(row) + [None] * 13)[:13]
        beleg_id       = clean(cells[0])   # identifier
        quelle         = clean(cells[1])   # source reference (Markdown link with interview ID)
        interview_id   = clean(cells[2])   # interview archive_id
        # cells[3]: interview_datum (informational only)
        quelle_sprecher = clean(cells[4])  # speaker person identifier → Interview.interviewee
        betrifft       = clean(cells[5])   # people mentioned (person identifiers)
        timecode       = clean(cells[6])   # timecode
        themen         = clean(cells[7])   # topics/concepts
        zitat          = clean(cells[8])   # quote
        markierung     = clean(cells[9])   # classification
        # cells[10]: event_ids (not yet linked — events must be imported first)
        event_confidence = clean(cells[11])
        notizen        = clean(cells[12])  # notes

        if is_empty(beleg_id):
            continue

        # Check if already imported
        existing = api_get("extractions", params={"search": beleg_id})
        if any(e["identifier"] == beleg_id for e in existing):
            continue

        # Resolve person mentioned (first identifier from comma-separated list)
        person_id = None
        if betrifft:
            first_person_id = betrifft.split(",")[0].strip()
            person_id = get_person_id(first_person_id)

        # Resolve interview (falls back to quelle column if interview_id is empty)
        interview_db_id = get_interview_id(interview_id, quelle=quelle)

        # Link quelle_sprecher → Person → Interview.interviewee
        if not is_empty(quelle_sprecher) and interview_db_id:
            speaker_person_id = get_person_id(quelle_sprecher)
            if speaker_person_id:
                try:
                    api_patch("interviews", interview_db_id, {"interviewee": speaker_person_id})
                except Exception as e:
                    errors.append(f"interviewee {interview_id}: {e}")

        # Resolve all concepts from comma-separated themen
        concept_ids = []
        if themen:
            for topic in themen.split(","):
                topic = topic.strip()
                if not is_empty(topic):
                    cid = get_or_create_concept(topic)
                    if cid:
                        concept_ids.append(cid)

        payload = {
            "identifier": beleg_id,
            "timecode": timecode,
            "quote": zitat,
            "classification": markierung,
            "event_confidence": event_confidence,
            "notes": notizen,
        }
        # Detect language from quote text (only if long enough and confidence > 70%)
        if len(zitat) >= 20:
            try:
                lang_result = detect_language(zitat)
                if lang_result and lang_result[0]["score"] > 0.7:
                    payload["language"] = lang_result[0]["lang"]
            except Exception:
                pass
        if person_id:
            payload["people_mentioned"] = person_id
        if interview_db_id:
            payload["interview"] = interview_db_id
        if concept_ids:
            payload["concepts"] = concept_ids

        try:
            created = api_post("extractions", payload)
            results.append(created["id"])
        except Exception as e:
            errors.append(f"{beleg_id}: {e}")

    if errors:
        print(f"\n⚠ {len(errors)} errors:")
        for err in errors:
            print(f"  ❌ {err}")
    print(f"✅ Imported {len(results)} extractions from {xlsx_path.name}")
    return results

## Import all belege files

In [6]:
def import_all_belege(directory=None):
    """Find and import all belege_*.xlsx files from the data directory."""
    if directory is None:
        directory = DATA_DIR
    directory = Path(directory)
    all_results = []
    for xlsx_path in sorted(directory.rglob("belege_*.xlsx")):
        print(f"\nProcessing: {xlsx_path.name}")
        try:
            ids = import_belege(xlsx_path)
            all_results.extend(ids)
        except Exception as e:
            print(f"\u274c ERROR: {e}")
    print(f"\n\u2705 Total imported: {len(all_results)} extractions")
    return all_results

## Run the import

Uncomment the appropriate line below.

In [7]:
# Single file:
# result = import_belege(DATA_DIR / "stern_IS_S_00142" / "belege_josef_stern_v2.xlsx")

# All belege files:
result = import_all_belege()
len(result)


Processing: belege_template.xlsx


belege_template: 0row [00:00, ?row/s]

belege_template: 0row [00:00, ?row/s]

✅ Imported 0 extractions from belege_template.xlsx

Processing: belege_charlotte_bruenn_v2.xlsx


belege_charlotte_bruenn_v2:   0%|                      | 0/130 [00:00<?, ?row/s]

...

belege_josef_stern_v2:  96%|████████████████▎| 125/130 [00:08<00:00, 15.15row/s]

belege_josef_stern_v2:  98%|████████████████▋| 128/130 [00:08<00:00, 16.22row/s]

belege_josef_stern_v2: 100%|█████████████████| 130/130 [00:08<00:00, 15.57row/s]

✅ Imported 0 extractions from belege_josef_stern_v2.xlsx

✅ Total imported: 35 extractions


35